# JSBSim C172 Manual c172x do_trim Seed v46

This notebook tests the manual/reference C172 cruise initialization found in JSBSim mailing-list/examples: `c172x`, `ubody=160 ft/s`, `altitude=8000 ft`, throttle/mixture set before longitudinal `do_trim(0)`. It is intentionally minimal and exists to check whether the official example-style condition trims while our previous `c172p` 3000 ft setup did not.

## 0. Install & Imports

In [1]:
!pip install jsbsim -q
print('Install complete')


Install complete


In [2]:
import os, time, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jsbsim

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)
    COLAB_DRIVE_MOUNTED = False

FPS_PER_KT = 1.687809857


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Manual Reference Configuration

In [3]:
EXPERIMENT_NAME = 'jsbsim_c172_manual_c172x_dotrim_seed_v46'
RUN_MODE = 'manual_c172x_seed'

# Manual/example-derived values.
AIRCRAFT_CANDIDATES = ['c172x', 'c172p']  # c172x first, c172p fallback/check.
ALTITUDE_GRID_FT = [8000.0, 4000.0, 3000.0]
UBODY_FPS_GRID = [160.0, 168.78, 180.0]  # 160 fps ~= 94.8 kt; 168.78 fps = 100 kt.
THROTTLE_GRID = [1.0, 0.85, 0.72]
MIXTURE_GRID = [0.85, 0.87, 1.0]
PITCH_GRID_DEG = [0.0, -1.0, 1.0]
ELEVATOR_SEED_GRID = [0.0]
TRIM_MODE = 0
DT = 0.02
HOLD_SECONDS = 60.0
TAIL_SECONDS = 10.0

RESULT_ROOT = '/content/drive/MyDrive/Colab Result' if COLAB_DRIVE_MOUNTED else './Colab Result'
SAVE_DIR = os.path.join(RESULT_ROOT, 'PINN_MPC', EXPERIMENT_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)
print('SAVE_DIR:', SAVE_DIR)
print('Manual primary seed: aircraft=c172x, ubody=160 fps, altitude=8000 ft, throttle=1.0, mixture=0.85, do_trim=0')


SAVE_DIR: /content/drive/MyDrive/Colab Result/PINN_MPC/jsbsim_c172_manual_c172x_dotrim_seed_v46
Manual primary seed: aircraft=c172x, ubody=160 fps, altitude=8000 ft, throttle=1.0, mixture=0.85, do_trim=0


## 2. JSBSim Helpers

In [4]:
def get_prop(fdm, name, default=np.nan):
    try:
        return float(fdm[name])
    except Exception:
        return float(default)


def safe_set(fdm, name, value):
    try:
        fdm[name] = float(value)
        return True
    except Exception:
        return False


def make_fdm_manual(aircraft, altitude_ft, ubody_fps, pitch_deg, elevator_seed, throttle, mixture):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model(aircraft)
    # Try to mirror reset01.xml style. Some wheels accept different IC aliases,
    # so set ubody and vt aliases consistently.
    safe_set(fdm, 'ic/h-sl-ft', altitude_ft)
    safe_set(fdm, 'ic/altitude-ft', altitude_ft)
    safe_set(fdm, 'ic/u-fps', ubody_fps)
    safe_set(fdm, 'ic/ubody-fps', ubody_fps)
    safe_set(fdm, 'ic/vt-fps', ubody_fps)
    safe_set(fdm, 'ic/vt-kts', ubody_fps / FPS_PER_KT)
    safe_set(fdm, 'ic/theta-deg', pitch_deg)
    safe_set(fdm, 'ic/psi-true-deg', 200.0)
    safe_set(fdm, 'ic/long-gc-deg', 122.0)
    safe_set(fdm, 'ic/lat-gc-deg', 47.0)
    fdm.run_ic()
    # Engine/control setup similar to the c172 cruise script.
    safe_set(fdm, 'propulsion/engine/set-running', 1)
    safe_set(fdm, 'propulsion/magneto_cmd', 3)
    safe_set(fdm, 'propulsion/starter_cmd', 1)
    safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
    safe_set(fdm, 'fcs/mixture-cmd-norm', mixture)
    safe_set(fdm, 'fcs/elevator-cmd-norm', elevator_seed)
    return fdm


def read_state(fdm):
    vt_fps = get_prop(fdm, 'velocities/vt-fps')
    return {
        'alt_ft': get_prop(fdm, 'position/h-sl-ft'),
        'speed_fps': vt_fps,
        'speed_kts': vt_fps / FPS_PER_KT if np.isfinite(vt_fps) else np.nan,
        'theta_deg': np.degrees(get_prop(fdm, 'attitude/theta-rad')),
        'q_deg_s': np.degrees(get_prop(fdm, 'velocities/q-rad_sec')),
        'alpha_deg': np.degrees(get_prop(fdm, 'aero/alpha-rad', np.nan)),
        'elevator': get_prop(fdm, 'fcs/elevator-cmd-norm'),
        'throttle': get_prop(fdm, 'fcs/throttle-cmd-norm'),
        'mixture': get_prop(fdm, 'fcs/mixture-cmd-norm'),
    }


def step_hold(fdm, elevator, throttle, mixture):
    safe_set(fdm, 'propulsion/engine/set-running', 1)
    safe_set(fdm, 'fcs/elevator-cmd-norm', elevator)
    safe_set(fdm, 'fcs/throttle-cmd-norm', throttle)
    safe_set(fdm, 'fcs/mixture-cmd-norm', mixture)
    fdm.run()


def hold_existing(fdm, seconds=HOLD_SECONDS, label='hold'):
    steps = max(1, int(seconds / DT))
    rows=[]
    s0 = read_state(fdm)
    elevator = s0['elevator']; throttle = s0['throttle']; mixture = s0['mixture']
    for k in range(steps):
        s = read_state(fdm)
        s.update({'time_s': k * DT, 'label': label, 'cmd_elevator': elevator, 'cmd_throttle': throttle, 'cmd_mixture': mixture})
        rows.append(s)
        step_hold(fdm, elevator, throttle, mixture)
    return pd.DataFrame(rows)


def summarize_hold(df):
    tail_n = max(1, int(TAIL_SECONDS / DT))
    tail = df.tail(tail_n)
    tail_vs = (tail['alt_ft'].iloc[-1] - tail['alt_ft'].iloc[0]) / max(tail['time_s'].iloc[-1] - tail['time_s'].iloc[0], DT)
    return {
        'hold_alt_drift_ft': float(df['alt_ft'].iloc[-1] - df['alt_ft'].iloc[0]),
        'hold_speed_drift_kts': float(df['speed_kts'].iloc[-1] - df['speed_kts'].iloc[0]),
        'hold_tail_vs_fps': float(tail_vs),
        'hold_tail_q_rms_deg_s': float(np.sqrt(np.mean(tail['q_deg_s'].values ** 2))),
        'hold_max_alpha_abs_deg': float(np.max(np.abs(df['alpha_deg'].values))),
        'hold_final_pitch_deg': float(df['theta_deg'].iloc[-1]),
    }


## 3. Manual do_trim Seed Sweep

In [5]:
def run_one_manual_seed(aircraft, altitude_ft, ubody_fps, pitch_deg, elevator_seed, throttle, mixture):
    row = {
        'aircraft': aircraft,
        'seed_altitude_ft': float(altitude_ft),
        'seed_ubody_fps': float(ubody_fps),
        'seed_speed_kts': float(ubody_fps / FPS_PER_KT),
        'seed_pitch_deg': float(pitch_deg),
        'seed_elevator': float(elevator_seed),
        'seed_throttle': float(throttle),
        'seed_mixture': float(mixture),
        'trim_mode': TRIM_MODE,
        'success': False,
        'status': 'not_run',
        'exception_type': '',
        'exception_message': '',
    }
    try:
        fdm = make_fdm_manual(aircraft, altitude_ft, ubody_fps, pitch_deg, elevator_seed, throttle, mixture)
        before = read_state(fdm)
        result = fdm.do_trim(int(TRIM_MODE))
        row['success'] = True
        row['status'] = f'returned:{result}'
        after = read_state(fdm)
        row.update({f'before_{k}': v for k, v in before.items()})
        row.update({f'trim_{k}': v for k, v in after.items()})
        hold_df = hold_existing(fdm, HOLD_SECONDS, label=f'{aircraft}_manual_seed')
        row.update(summarize_hold(hold_df))
    except Exception as exc:
        row['success'] = False
        row['status'] = 'failed'
        row['exception_type'] = type(exc).__name__
        row['exception_message'] = str(exc)
    return row

rows=[]
total = len(AIRCRAFT_CANDIDATES) * len(ALTITUDE_GRID_FT) * len(UBODY_FPS_GRID) * len(PITCH_GRID_DEG) * len(ELEVATOR_SEED_GRID) * len(THROTTLE_GRID) * len(MIXTURE_GRID)
print('manual do_trim seed cases:', total)
idx=0
for aircraft in AIRCRAFT_CANDIDATES:
    for altitude_ft in ALTITUDE_GRID_FT:
        for ubody_fps in UBODY_FPS_GRID:
            for pitch_deg in PITCH_GRID_DEG:
                for elevator_seed in ELEVATOR_SEED_GRID:
                    for throttle in THROTTLE_GRID:
                        for mixture in MIXTURE_GRID:
                            idx += 1
                            rows.append(run_one_manual_seed(aircraft, altitude_ft, ubody_fps, pitch_deg, elevator_seed, throttle, mixture))
                            if idx % 20 == 0:
                                tmp = pd.DataFrame(rows)
                                print(f'{idx}/{total} success={int(tmp.success.sum())}')

manual_df = pd.DataFrame(rows)
print('Success count:', int(manual_df.success.sum()), '/', len(manual_df))
display(manual_df.success.value_counts(dropna=False).to_frame('count'))
if manual_df.success.any():
    print('Successful manual seed trims:')
    cols = [c for c in manual_df.columns if c.startswith('seed_') or c.startswith('trim_') or c.startswith('hold_') or c in ['aircraft','success','status']]
    display(manual_df[manual_df.success][cols].sort_values('hold_tail_vs_fps', key=lambda s: np.abs(s)).head(20))
else:
    print('No success. Failure types:')
    display(manual_df.groupby(['aircraft','exception_type','exception_message']).size().reset_index(name='count').sort_values('count', ascending=False))


manual do_trim seed cases: 486


     JSBSim Flight Dynamics Model v1.3.0 Apr  9 2026 10:00:08
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

20/486 success=20
40/486 success=40
60/486 success=60
80/486 success=80
100/486 success=100
120/486 success=120
140/486 success=140
160/486 success=160
180/486 success=180
200/486 success=200
220/486 success=220
240/486 success=240
260/486 success=260
280/486 success=280
300/486 success=300
320/486 success=320
340/486 success=340
360/486 success=360
380/486 success=380
400/486 success=400
420/486 success=420
440/486 success=440
460/486 success=460
480/486 success=480
Success count: 486 / 486


,count
success,
True,486


Successful manual seed trims:


,aircraft,seed_altitude_ft,seed_ubody_fps,seed_speed_kts,seed_pitch_deg,seed_elevator,seed_throttle,seed_mixture,trim_mode,success,...,trim_alpha_deg,trim_elevator,trim_throttle,trim_mixture,hold_alt_drift_ft,hold_speed_drift_kts,hold_tail_vs_fps,hold_tail_q_rms_deg_s,hold_max_alpha_abs_deg,hold_final_pitch_deg
483,c172p,3000.0,180.00,106.647084,1.0,0.0,0.72,0.85,0,True,...,0.247987,0.0,0.782674,0.85,-150.895504,13.904365,-6.348335,3.386758,0.251540,-3.955572
480,c172p,3000.0,180.00,106.647084,1.0,0.0,0.85,0.85,0,True,...,0.247987,0.0,0.782674,0.85,-150.895504,13.904365,-6.348335,3.386758,0.251540,-3.955572
477,c172p,3000.0,180.00,106.647084,1.0,0.0,1.00,0.85,0,True,...,0.247987,0.0,0.782674,0.85,-150.895504,13.904365,-6.348335,3.386758,0.251540,-3.955572
484,c172p,3000.0,180.00,106.647084,1.0,0.0,0.72,0.87,0,True,...,0.247986,0.0,0.775200,0.87,-150.901032,13.897563,-6.348925,3.386173,0.251540,-3.956352
481,c172p,3000.0,180.00,106.647084,1.0,0.0,0.85,0.87,0,True,...,0.247986,0.0,0.775200,0.87,-150.901032,13.897563,-6.348925,3.386173,0.251540,-3.956352
478,c172p,3000.0,180.00,106.647084,1.0,0.0,1.00,0.87,0,True,...,0.247986,0.0,0.775200,0.87,-150.901032,13.897563,-6.348925,3.386173,0.251540,-3.956352
485,c172p,3000.0,180.00,106.647084,1.0,0.0,0.72,1.00,0,True,...,0.247987,0.0,0.749039,1.00,-151.037766,13.923475,-6.349929,3.387200,0.251566,-3.954304
479,c172p,3000.0,180.00,106.647084,1.0,0.0,1.00,1.00,0,True,...,0.247987,0.0,0.749039,1.00,-151.037766,13.923475,-6.349929,3.387200,0.251566,-3.954304
482,c172p,3000.0,180.00,106.647084,1.0,0.0,0.85,1.00,0,True,...,0.247987,0.0,0.749039,1.00,-151.037766,13.923475,-6.349929,3.387200,0.251566,-3.954304
437,c172p,3000.0,168.78,99.999416,0.0,0.0,0.85,1.00,0,True,...,0.751325,0.0,0.688373,1.00,-195.381889,12.495236,-6.352193,3.344492,0.755286,-3.814622


## 4. Save

In [6]:
csv_path = os.path.join(SAVE_DIR, f'manual_c172x_dotrim_seed_v46_{RUN_MODE}.csv')
manual_df.to_csv(csv_path, index=False)
print('Saved:', csv_path)

summary_path = os.path.join(SAVE_DIR, f'manual_c172x_dotrim_seed_summary_v46_{RUN_MODE}.csv')
manual_df.groupby(['aircraft','success','exception_type','exception_message'], dropna=False).size().reset_index(name='count').to_csv(summary_path, index=False)
print('Saved summary:', summary_path)


Saved: /content/drive/MyDrive/Colab Result/PINN_MPC/jsbsim_c172_manual_c172x_dotrim_seed_v46/manual_c172x_dotrim_seed_v46_manual_c172x_seed.csv
Saved summary: /content/drive/MyDrive/Colab Result/PINN_MPC/jsbsim_c172_manual_c172x_dotrim_seed_v46/manual_c172x_dotrim_seed_summary_v46_manual_c172x_seed.csv


## 5. Notes

If this succeeds for `c172x` but not for `c172p`, the issue is model mismatch: the JSBSim cruise/trim references are for `c172x`, while earlier notebooks used `c172p`. If even the primary manual seed fails, inspect whether the Python wheel includes a materially different C172 model or whether the engine-start properties differ from the standalone script behavior.